<p align="right"><a href="./C2_W4_Lab_02_Tree_Ensemble.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab - 树集成（Trees Ensemble）

在本 notebook 中，你将：

 - 用 Pandas 对数据集做 one-hot 编码
 - 用 scikit-learn 实现决策树（Decision Tree）、随机森林（Random Forest）和 XGBoost 模型

我们来导入你会用到的库。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
!pip install xgboost --quiet
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

RANDOM_STATE = 55 ## You will pass it to every sklearn call so we ensure reproducibility

# 1. 加载数据集

来自 [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)

背景
心血管疾病（CVDs）是全球头号死因，每年约夺走 1790 万人的生命，占全球所有死亡的 31%。心力衰竭是 CVDs 引起的常见事件，本数据集包含 11 个可用于预测潜在心脏病的特征。

患有心血管疾病、或处于高心血管风险的人需要早期发现和管理，而机器学习模型在此能提供很大帮助。

你将开发模型，根据下面所有信息，预测某个人患心血管疾病的可能性有多大。

#### 属性说明
- Age：患者年龄 [岁]
- Sex：患者性别 [M: 男, F: 女]
- ChestPainType：胸痛类型 [TA: 典型心绞痛, ATA: 非典型心绞痛, NAP: 非心绞痛性疼痛, ASY: 无症状]
- RestingBP：静息血压 [mm Hg]
- Cholesterol：血清胆固醇 [mm/dl]
- FastingBS：空腹血糖 [1: 若 FastingBS > 120 mg/dl, 0: 否则]
- RestingECG：静息心电图结果 [Normal: 正常, ST: 有 ST-T 波异常, LVH: 按 Estes 标准显示可能或明确的左心室肥大]
- MaxHR：达到的最大心率 [60 到 202 之间的数值]
- ExerciseAngina：运动诱发的心绞痛 [Y: 是, N: 否]
- Oldpeak：oldpeak = ST [以压低度量的数值]
- ST_Slope：运动峰值 ST 段的斜率 [Up: 上斜, Flat: 平坦, Down: 下斜]
- HeartDisease：输出类别 [1: 有心脏病, 0: 正常]

现在加载数据集。如上所示，以下变量：

- Sex
- ChestPainType
- RestingECG
- ExerciseAngina
- ST_Slope

是 *类别型（categorical）* 的，所以你必须对它们做 one-hot 编码。

In [ ]:
# Load the dataset using pandas
df = pd.read_csv("heart.csv")

In [ ]:
df.head()

在用模型之前，你必须做一些数据工程。有 5 个类别型特征，所以你会用 Pandas 对它们做 one-hot 编码。

## 2. 用 Pandas 做 one-hot 编码

首先你会移除二值变量，因为对它们做 one-hot 编码没有意义。为此，你只需数一数每个类别型变量有多少个不同取值，只考虑取值有 3 个或以上的变量。

In [ ]:
cat_variables = ['Sex',
'ChestPainType',
'RestingECG',
'ExerciseAngina',
'ST_Slope'
]

提醒一下，one-hot 编码的目的是把一个有 `n` 种输出的类别型变量变成 `n` 个二值变量。

Pandas 有一个内置方法来做 one-hot 编码，即函数 `pd.get_dummies`。这个函数有几个参数，这里你只用到几个：

 - data：要使用的 DataFrame
 - prefix：一个前缀列表，让你知道处理的是哪个值
 - columns：要做 one-hot 编码的列的列表。'prefix' 和 'columns' 必须长度相同。

想了解更多，你随时可以输入 `help(pd.get_dummies)` 阅读该函数的完整文档。

In [ ]:
# This will replace the columns with the one-hot encoded ones and keep the columns outside 'columns' argument as it is.
df = pd.get_dummies(data = df,
                         prefix = cat_variables,
                         columns = cat_variables)

In [ ]:
df.head()

现在你会定义本实验要构建的模型将使用的最终变量集。

In [ ]:
var = [x for x in df.columns if x not in 'HeartDisease'] ## Removing our target variable

注意变量数量的变化。你从 11 个变量开始，现在有：

In [ ]:
print(len(var))

# 3. 拆分数据集

本节你会把数据集拆成训练集和测试集。你会用 Scikit-learn 的 `train_test_split` 函数。我们先看看它的参数。

In [ ]:
help(train_test_split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[var], df['HeartDisease'], train_size = 0.8, random_state = RANDOM_STATE)

# We will keep the shuffle = True since our dataset has not any time dependency.

In [ ]:
print(f'train samples: {len(X_train)}\ntest samples: {len(X_test)}')
print(f'target proportion: {sum(y_train)/len(y_train):.4f}')

# 4. 构建模型

## 4.1 决策树（Decision Tree）

本节我们来用你之前学过的决策树，但现在用 [Scikit-learn 的实现](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)。

Scikit-learn 的决策树对象有若干超参数。你只会用其中一些，本实验也不会做特征选择或超参数调优（但鼓励你自己去做并比较结果 :-) ）

你这里会用到并考察的超参数是：

 - min_samples_split：分裂一个内部节点所需的最小样本数。这可能防止过拟合。
 - max_depth：树的最大深度。这可能防止过拟合。

In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700] ## If the number is an integer, then it is the actual quantity of samples,
max_depth_list = [1,2, 3, 4, 8, 16, 32, 64, None] # None means that there is no depth limit.

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

注意增大 `min_samples_split` 会减少过拟合。

我们对 `max_depth` 做同样的实验。

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = DecisionTreeClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

测试准确率在 tree_depth=3 时最高。当允许的深度更小时，树无法做足够的分裂来区分正负样本（有欠拟合问题）；但当允许的深度太高（>= 5）时，树变得过于专门化于训练集，从而在测试集上损失准确率（有过拟合问题）。于是我们最终的树模型会用：

- `max_depth = 3`
- `min_samples_split = 50`

In [ ]:
decision_tree_model = DecisionTreeClassifier(min_samples_split = 50,
                                             max_depth = 3,
                                             random_state = RANDOM_STATE).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(decision_tree_model.predict(X_test),y_test):.4f}")

没有过拟合的迹象，尽管这些指标并不算很好。

## 4.2 随机森林（Random Forest）

现在我们也来试试随机森林算法，同样用 Scikit-learn 的实现。自然地，上面所有超参数在这个算法里都存在——因为它就是决策树的集成——但它还多一个你会用到的超参数 `n_estimators`，即会拟合多少棵不同的决策树。

记住，对随机森林，训练每棵树时你都用随机选取的特征子集 *和* 训练集子集。此例中你会用课程里讲的特征数量，即 $\sqrt{n}$（$n$ 是特征数），不过这也可以修改。关于随机森林超参数的更多信息，你可以运行 `help(RandomForestClassifier)`。

另一个不影响最终结果、但能加速计算的参数叫 `n_jobs`。由于每棵树的拟合彼此独立，可以并行拟合。把 `n_jobs` 设高会增加它用多少 CPU 核心。注意，取非常接近你 CPU 最大核数的值可能影响整机性能、甚至导致卡死。

你会再次运行同样的脚本，但加一个参数 `n_estimators`，我们在 10、50、100 之间选择，默认是 100。

In [ ]:
min_samples_split_list = [2,10, 30, 50, 100, 200, 300, 700]  ## If the number is an integer, then it is the actual quantity of samples,
                                             ## If it is a float, then it is the percentage of the dataset
max_depth_list = [2, 4, 8, 16, 32, 64, None]
n_estimators_list = [10,50,100,500]

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for min_samples_split in min_samples_split_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(min_samples_split = min_samples_split,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('min_samples_split')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(min_samples_split_list )),labels=min_samples_split_list) 
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for max_depth in max_depth_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(max_depth = max_depth,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(max_depth_list )),labels=max_depth_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

In [ ]:
accuracy_list_train = []
accuracy_list_test = []
for n_estimators in n_estimators_list:
    # You can fit the model at the same time you define it, because the fit function returns the fitted estimator.
    model = RandomForestClassifier(n_estimators = n_estimators,
                                   random_state = RANDOM_STATE).fit(X_train,y_train) 
    predictions_train = model.predict(X_train) ## The predicted values for the train dataset
    predictions_test = model.predict(X_test) ## The predicted values for the test dataset
    accuracy_train = accuracy_score(predictions_train,y_train)
    accuracy_test = accuracy_score(predictions_test,y_test)
    accuracy_list_train.append(accuracy_train)
    accuracy_list_test.append(accuracy_test)

plt.title('Train x Test metrics')
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.xticks(ticks = range(len(n_estimators_list )),labels=n_estimators_list)
plt.plot(accuracy_list_train)
plt.plot(accuracy_list_test)
plt.legend(['Train','Test'])

接下来我们用如下参数拟合一个随机森林：

 - max_depth: 8
 - min_samples_split: 10
 - n_estimators: 100

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators = 100,
                                             max_depth = 8, 
                                             min_samples_split = 10).fit(X_train,y_train)

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(random_forest_model.predict(X_test),y_test):.4f}")

你已经演示了如何一个超参数一个超参数地寻找最佳值。但你不应忽视：在试某个超参数时，我们总要把其他超参数固定在某些默认值上，这让我们只能说清该超参数值相对于那些默认值的变化。原则上，如果在被调的 3 个超参数上各有 4 个候选值，你应有共 4 x 4 x 4 = 64 种组合，然而你这种做法只会给出 4 + 4 + 4 = 12 个结果。要试遍所有组合，可以用一个 sklearn 的实现 GridSearchCV；此外，它有个 refit 参数会自动在最佳组合上重新拟合模型，你就不必显式编程了。关于 GridSearchCV 的更多信息，请参考其[文档](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)。

## 4.3 XGBoost

现在，本实验你要测试的最后一个模型是梯度提升（Gradient Boosting）模型，叫 XGBoost。如你在课程里所见，提升（boosting）方法训练若干棵树，但它们不是彼此不相关，而是相继拟合以最小化误差。

这个模型包含的参数与任意决策树的参数相同，外加一些其他参数，如学习率——即 XGBoost 内部用来在每个训练步最小化误差的梯度下降方法的步长。

XGBoost 有个有趣的地方：在拟合时，它允许你传入一个形如 `(X_val,y_val)` 的评估数据集列表，在每次迭代它会在这些评估数据集上测量代价（或评估指标），一旦代价（或指标）在若干轮（称为 early_stopping_rounds）内不再下降，训练就停止。我们借此能自动控制多少个 estimator 就够了，也能避免因 estimator 过多导致的过拟合。

首先，我们来定义训练集的一个子集（这里不应使用测试集）。

In [ ]:
n = int(len(X_train)*0.8) ## Let's use 80% to train and 20% to eval

In [ ]:
X_train_fit, X_train_eval, y_train_fit, y_train_eval = X_train[:n], X_train[n:], y_train[:n], y_train[n:]

You can then set a large number of estimators, because you can stop if the cost function stops decreasing.

In [ ]:
xgb_model = XGBClassifier(n_estimators = 500, learning_rate = 0.1,verbosity = 1, random_state = RANDOM_STATE)
xgb_model.fit(X_train_fit,y_train_fit, eval_set = [(X_train_eval,y_train_eval)], early_stopping_rounds = 50)
# Here we must pass a list to the eval_set, because you can have several different tuples ov eval sets. The parameter 
# early_stopping_rounds is the number of iterations that it will wait to check if the cost function decreased or not.
# If not, it will stop and get the iteration that returned the lowest metric on the eval set.

可以看到，尽管你传入 500 个 estimator 去拟合，算法只拟合了 66 个，因为用来度量训练轮次的 log-loss 开始上升了。事实上，estimator 数量甚至少于 66。如果你仔细看指标，会发现拟合 16 棵树时就达到了 log-loss 的最小值，而这确实是最终模型里拟合的树的数量：

In [ ]:
xgb_model.best_iteration

In [ ]:
print(f"Metrics train:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_train),y_train):.4f}\nMetrics test:\n\tAccuracy score: {accuracy_score(xgb_model.predict(X_test),y_test):.4f}")

你可以看到 RandomForest 达到了最好的准确率，但总体结果都很接近。而且注意，我们用 XGBoost 得到的测试指标与 RandomForest 非常接近，而我们甚至没做任何超参数搜索！XGBoost 的优势是它比随机森林更快，而且参数更多，因此你能对模型做更精细的调优、取得更好的结果。


恭喜，你已经学会用 scikit-learn 库的决策树、随机森林，以及 XGBoost！